# 01. 캐릭터 매핑 & 대시보드 전처리 (팀 공식 통합본)

**원본 위치**: `이터널리턴_코드_정리.ipynb` cell 0~8 (섹션: `# 역할군 생성`, `# 임원진, 실무자 전용 대시보드 전처리 코드`)

모든 후속 분석(`02_balance_isolation_forest_shap.ipynb`가 참고하는 파이프라인, K-means 뉴비 군집 등)이 이 노트북의 산출물
`EternalReturn_kakaogames_2024_character_added.csv`를 입력으로 사용합니다. 원본 셀 내용은 수정하지 않았고, 순서와 마크다운 헤더만 정리했습니다.

- 관련 문서: `../../docs/02_analysis_pipeline.md` [1], [2]절 / `../../docs/03_issues_and_troubleshooting.md` #13


## 1-1. 원본 로드

In [ ]:
import pandas as pd

input_path = "/content/EternalReturn_kakaogames_2024.csv"

df = pd.read_csv(input_path, low_memory=False)

print(df.shape)
print(df.columns.tolist())
df.head()

## 1-2. 캐릭터 매핑표 (74종, characterNum → 이름/역할군)

In [ ]:
#캐릭터 매핑표
character_info = {
    1: {"name_en": "Jackie", "name_kr": "재키", "role": "전사"},
    2: {"name_en": "Aya", "name_kr": "아야", "role": "스킬 딜러/원거리 딜러"},
    3: {"name_en": "Fiora", "name_kr": "피오라", "role": "전사"},
    4: {"name_en": "Magnus", "name_kr": "매그너스", "role": "탱커/전사"},
    5: {"name_en": "Zahir", "name_kr": "자히르", "role": "스킬 딜러"},
    6: {"name_en": "Nadine", "name_kr": "나딘", "role": "스킬 딜러/원거리 딜러"},
    7: {"name_en": "Hyunwoo", "name_kr": "현우", "role": "전사"},
    8: {"name_en": "Hart", "name_kr": "하트", "role": "원거리 딜러"},
    9: {"name_en": "Isol", "name_kr": "아이솔", "role": "스킬 딜러/원거리 딜러"},
    10: {"name_en": "Li Dailin", "name_kr": "리 다이린", "role": "전사"},
    11: {"name_en": "Yuki", "name_kr": "유키", "role": "전사"},
    12: {"name_en": "Hyejin", "name_kr": "혜진", "role": "스킬 딜러"},
    13: {"name_en": "Xiukai", "name_kr": "쇼우", "role": "탱커"},
    14: {"name_en": "Chiara", "name_kr": "키아라", "role": "전사"},
    15: {"name_en": "Sissela", "name_kr": "시셀라", "role": "스킬 딜러"},
    16: {"name_en": "Silvia", "name_kr": "실비아", "role": "전사/스킬 딜러"},
    17: {"name_en": "Adriana", "name_kr": "아드리아나", "role": "스킬 딜러"},
    18: {"name_en": "Shoichi", "name_kr": "쇼이치", "role": "암살자"},
    19: {"name_en": "Emma", "name_kr": "엠마", "role": "스킬 딜러"},
    20: {"name_en": "Lenox", "name_kr": "레녹스", "role": "탱커"},
    21: {"name_en": "Rozzi", "name_kr": "로지", "role": "원거리 딜러"},
    22: {"name_en": "Luke", "name_kr": "루크", "role": "전사"},
    23: {"name_en": "Cathy", "name_kr": "캐시", "role": "암살자"},
    24: {"name_en": "Adela", "name_kr": "아델라", "role": "스킬 딜러"},
    25: {"name_en": "Bernice", "name_kr": "버니스", "role": "원거리 딜러"},
    26: {"name_en": "Barbara", "name_kr": "바바라", "role": "스킬 딜러"},
    27: {"name_en": "Alex", "name_kr": "알렉스", "role": "전사"},
    28: {"name_en": "Sua", "name_kr": "수아", "role": "전사/스킬 딜러"},
    29: {"name_en": "Leon", "name_kr": "레온", "role": "전사"},
    30: {"name_en": "Eleven", "name_kr": "일레븐", "role": "탱커"},
    31: {"name_en": "Rio", "name_kr": "리오", "role": "원거리 딜러"},
    32: {"name_en": "William", "name_kr": "윌리엄", "role": "원거리 딜러"},
    33: {"name_en": "Nicky", "name_kr": "니키", "role": "전사"},
    34: {"name_en": "Nathapon", "name_kr": "나타폰", "role": "스킬 딜러"},
    35: {"name_en": "Jan", "name_kr": "얀", "role": "전사"},
    36: {"name_en": "Eva", "name_kr": "에바", "role": "스킬 딜러"},
    37: {"name_en": "Daniel", "name_kr": "다니엘", "role": "암살자"},
    38: {"name_en": "Jenny", "name_kr": "제니", "role": "원거리 딜러"},
    39: {"name_en": "Camilo", "name_kr": "카밀로", "role": "전사"},
    40: {"name_en": "Chloe", "name_kr": "클로에", "role": "원거리 딜러"},
    41: {"name_en": "Johann", "name_kr": "요한", "role": "지원가"},
    42: {"name_en": "Bianca", "name_kr": "비앙카", "role": "스킬 딜러"},
    43: {"name_en": "Celine", "name_kr": "셀린", "role": "스킬 딜러"},
    44: {"name_en": "Echion", "name_kr": "에키온", "role": "전사"},
    45: {"name_en": "Mai", "name_kr": "마이", "role": "탱커/지원가"},
    46: {"name_en": "Aiden", "name_kr": "에이든", "role": "전사"},
    47: {"name_en": "Laura", "name_kr": "라우라", "role": "전사"},
    48: {"name_en": "Tia", "name_kr": "띠아", "role": "스킬 딜러"},
    49: {"name_en": "Felix", "name_kr": "펠릭스", "role": "전사"},
    50: {"name_en": "Elena", "name_kr": "엘레나", "role": "탱커"},
    51: {"name_en": "Priya", "name_kr": "프리야", "role": "스킬 딜러/지원가"},
    52: {"name_en": "Adina", "name_kr": "아디나", "role": "스킬 딜러/지원가"},
    53: {"name_en": "Markus", "name_kr": "마커스", "role": "탱커/전사"},
    54: {"name_en": "Karla", "name_kr": "칼라", "role": "원거리 딜러"},
    55: {"name_en": "Estelle", "name_kr": "에스텔", "role": "탱커/지원가"},
    56: {"name_en": "Piolo", "name_kr": "피올로", "role": "전사"},
    57: {"name_en": "Martina", "name_kr": "마르티나", "role": "원거리 딜러"},
    58: {"name_en": "Haze", "name_kr": "헤이즈", "role": "스킬 딜러"},
    59: {"name_en": "Isaac", "name_kr": "아이작", "role": "전사"},
    60: {"name_en": "Tazia", "name_kr": "타지아", "role": "스킬 딜러"},
    61: {"name_en": "Irem", "name_kr": "이렘", "role": "전사/스킬 딜러"},
    62: {"name_en": "Theodore", "name_kr": "테오도르", "role": "원거리 딜러/지원가"},
    63: {"name_en": "Ly Anh", "name_kr": "이안", "role": "전사"},
    64: {"name_en": "Vanya", "name_kr": "바냐", "role": "전사"},
    65: {"name_en": "Debi & Marlene", "name_kr": "데비&마를렌", "role": "전사"},
    66: {"name_en": "Arda", "name_kr": "아르다", "role": "스킬 딜러/지원가"},
    67: {"name_en": "Abigail", "name_kr": "아비게일", "role": "전사"},
    68: {"name_en": "Alonso", "name_kr": "알론소", "role": "탱커"},
    69: {"name_en": "Leni", "name_kr": "레니", "role": "지원가"},
    70: {"name_en": "Tsubame", "name_kr": "츠바메", "role": "원거리 딜러"},
    71: {"name_en": "Kenneth", "name_kr": "케네스", "role": "전사"},
    72: {"name_en": "Katja", "name_kr": "카티야", "role": "원거리 딜러"},
    73: {"name_en": "Charlotte", "name_kr": "샬럿", "role": "지원가"},
    74: {"name_en": "Darko", "name_kr": "다르코", "role": "전사"},
}

## 1-3. 매핑 적용

In [ ]:
# characterNum이 숫자가 아닐 경우를 대비해서 숫자형으로 변환
df["characterNum"] = pd.to_numeric(df["characterNum"], errors="coerce")

df["character_name_en"] = df["characterNum"].map(
    lambda x: character_info.get(x, {}).get("name_en")
)

df["character_name_kr"] = df["characterNum"].map(
    lambda x: character_info.get(x, {}).get("name_kr")
)

df["character_role"] = df["characterNum"].map(
    lambda x: character_info.get(x, {}).get("role")
)

df[["characterNum", "character_name_en", "character_name_kr", "character_role"]].head(10)

## 1-4. 매핑 검증 (트러블슈팅 #13)

`characterNum`이 매핑 딕셔너리 범위 밖이면 `character_name_kr`이 조용히 NaN이 되고, 이후 groupby에서 해당 유저 기록이
에러 없이 통째로 누락됩니다. 그래서 매핑 후 반드시 누락 여부를 확인합니다.

In [ ]:
unmapped = df[df["character_name_kr"].isna()]["characterNum"].dropna().unique()

print("매핑 안 된 characterNum:")
print(sorted(unmapped))

## 1-5. 저장 (다운스트림 분석의 공통 입력 파일)

In [ ]:
output_path = "/content/EternalReturn_kakaogames_2024_character_added.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

## 1-6. (Colab 전용) 로컬 다운로드

로컬/서버 환경에서는 이 셀을 건너뛰고 파일을 직접 저장 경로에 두면 됩니다.

In [ ]:
from google.colab import files

files.download(output_path)

## 1-7. 역할별 대시보드 컬럼 필터링

228개 원본 컬럼 중 임원진/기획자/밸런스팀/사업팀이 실제로 쓰는 컬럼만 추린 리스트. 이 컬럼 그룹 구조가 곧 최종 슬라이드의 '분석 범위' 정의와 대응됩니다.

In [ ]:
import pandas as pd

keep_columns = [
# 기본 컬럼
"matchingTeamMode", "characterLevel", "serverName","gameId","userNum",
"premadeMatchingType", "botAdded",
"characterNum",
"bestWeapon", "gameRank", "victory", "playerKill",
"rankPoint", "damageToPlayer", "damageToPlayer_skill", "damageToPlayer_basic",
"startDtm", "versionMajor", "versionMinor",
# 밸런스팀 보조
"playerAssistant", "teamKill", "totalDoubleKill", "totalTripleKill",
"killDetails", "ccTimeToPlayer", "healAmount", "characterLevel",
"bestWeaponLevel", "tacticalSkillGroup", "traitFirstCore",
"matchingTeamMode", "serverName",
# 서비스기획자 핵심
"playTime", "survivableTime", "placeOfStart", "placeOfDeath",
"useHyperLoop", "useSecurityConsole", "accountLevel", "victory",
"gameRank", "startDtm", "serverName","causeOfDeath",
"craftUncommon", "craftRare", "craftEpic", "craftLegend",
# 서비스기획자 보조
"useReconDrone", "addSurveillanceCamera", "routeIdOfStart", "monsterKill",
"playerKill", "killerCharacter", "creditRevivalCount",
"crGetAnimal", "giveUp", "teamSpectator", "matchingTeamMode",
# 사업팀 보조
"mmrGainInGame"
]

keep_columns = list(dict.fromkeys(keep_columns)) # 중복 제거

df_filtered = df[keep_columns]
print(f"필터링 후 컬럼 수: {len(keep_columns)}")
print(df_filtered)

## 다음 단계

- 밸런스팀 관점 분석(Isolation Forest + SHAP) → `individual/01_balance_isolation_forest_shap.ipynb`
- 기획자 관점 뉴비 정의 + 행동 군집(K-means) → `individual/02_planner_newbie_kmeans_clustering.ipynb`

두 분석 모두 이 노트북에서 만든 `EternalReturn_kakaogames_2024_character_added.csv`를 입력으로 사용합니다.